In [ ]:
#Mount Google Drive so TF has access to the dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Import the required libraries
import tensorflow as tf
import numpy as np
import os

In [ ]:
#Path to the dataset folder inside Google Drive
dataset_path = "/content/drive/MyDrive/dataset2"

In [ ]:
#Print dataset folders to show the class folders
print(os.listdir(dataset_path))

['Plastic', 'Cardboard', 'Metal']


In [ ]:
#Load dataset from directory

#TensorFlow automatically assign labels based on folder names
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,          # dataset location
    validation_split=0.2,  # 20% of images used for validaiton
    subset="training",     # this portion is used for training
    seed=123,              # ensure a reproducible split
    image_size=(96, 96),   # resize images to 96x96
    batch_size=32          # number of images per batch
)

#Load validation dataset
val_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",    # this portion is used for validaiton
    seed=123,
    image_size=(96, 96),
    batch_size=32
)

#Prints class name and number of images in total and for each subset
class_names = train_dataset.class_names
print("Classes:", class_names)

Found 61 files belonging to 3 classes.
Using 49 files for training.
Found 61 files belonging to 3 classes.
Using 12 files for validation.
Classes: ['Cardboard', 'Metal', 'Plastic']


In [ ]:
#Normalize pixel values

#ORIGINAL PIXELS RANGE: 0-255
#AFTER RESCALING: 0-1
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
val_dataset = val_dataset.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
#Performance optimization for TensorFlow data pipeline
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
#Data augmentation layer (Unused)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),  # Randomly flip images horizontally
    tf.keras.layers.RandomRotation(0.05),      # Randomly rotates images slightly
    tf.keras.layers.RandomZoom(0.1),           # Randomly zoom images slightly
])

In [ ]:
#Import pre-trained MobileNetV2 model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

#Load MobileNetV2 pre-trained on ImageNet
base_model = MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,        # include_top=False to remove its original classifier
    weights='imagenet',
    alpha=0.35
)

#Freeze pretrained layers so their weights are not updated
#Prevents overfitting since dataset is small
base_model.trainable = False

#Build new classification model using MobileNetV2 features
model = models.Sequential([
    base_model,                            # Pretrained feature extractor
    layers.GlobalAveragePooling2D(),       # Convert feature maps to a single vector
    layers.Dense(64, activation='relu'),   # Fully connected layer for learning material features
    layers.Dropout(0.3),                   # Dropout to reduce overfitting
    layers.Dense(3, activation='softmax')  # Final classification layer (3 classes)
])

2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
#Compile model with adam optimizer
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy', # Labels are integers
    metrics=['accuracy']
)

In [ ]:
#Early stopping callback
#Stops training when validation loss stops improving
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',      # Watch validation loss
    patience=5,              # Wait 5 epochs before stopping
    restore_best_weights=True
)

In [ ]:
#Train model
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    callbacks=[early_stop]
)

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 19s 4s/step - accuracy: 0.5102 - loss: 1.3687 - val_accuracy: 0.4167 - val_loss: 1.6222
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 378ms/step - accuracy: 0.6735 - loss: 0.8190 - val_accuracy: 0.2500 - val_loss: 1.3484
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 302ms/step - accuracy: 0.6735 - loss: 0.7189 - val_accuracy: 0.3333 - val_loss: 1.1613
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 400ms/step - accuracy: 0.7959 - loss: 0.5599 - val_accuracy: 0.3333 - val_loss: 1.1334
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step - accuracy: 0.7959 - loss: 0.4165 - val_accuracy: 0.4167 - val_loss: 1.1472
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 209ms/step - accuracy: 0.9184 - loss: 0.3235 - val_accuracy: 0.4167 - val_loss: 1.1640
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step - accuracy: 0.8571 - loss: 0.3220 - val_accuracy: 0.4167 - val_loss: 1.1652
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 383ms/step - accuracy: 0.8367 - loss: 0.3208 - val_accuracy: 0.4167 - val_loss: 1

In [ ]:
#Saves trained model into .keras format
model.save("waste_classifier.keras")

In [ ]:
#File conversion

#Load and convert
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# No optimizations — pure float32
tflite_model = converter.convert()

#save the .tflite file
with open('waste_classifier.tflite', 'wb') as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmp5mvdi03j'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 96, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  136747920995152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136747920998032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136747920997840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136747920997264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136747920995728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136747920997072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136747920996112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136747918459728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136747920997648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136747920998224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136747918460

In [ ]:
with open('waste_classifier.tflite', 'rb') as f:
    data = f.read()

with open('waste_classifier.h', 'w') as f:
    f.write('#pragma once\n\n')
    f.write(f'const unsigned int waste_classifier_tflite_len = {len(data)};\n\n')
    f.write('alignas(8) const unsigned char waste_classifier_tflite[] = {\n  ')
    hex_values = [f'0x{b:02x}' for b in data]
    lines = [', '.join(hex_values[i:i+12]) for i in range(0, len(hex_values), 12)]
    f.write(',\n  '.join(lines))
    f.write('\n};\n')

print(f"Done! Size: {len(data)} bytes")

Done! Size: 1922460 bytes


In [ ]:
#Final conversion to convert tflite file into header format for Arduino IDE
#!xxd -i waste_classifier.tflite > waste_classifier.h

In [ ]:
#Testing the model using a single image each time

from tensorflow.keras.utils import load_img, img_to_array
import numpy as np

img_path = "/content/CBT_1.jpg"  # Image path

# Load and preprocess
img = load_img(img_path, target_size=(96, 96))
img_array = img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
prediction = model.predict(img_array)

# Map to class name
predicted_class = class_names[np.argmax(prediction)]
print("Predicted class:", predicted_class)
print("Classes:", class_names)
print("Raw prediction:", prediction)
print("Confidence:", np.max(prediction))

In [ ]:
!grep "len" waste_classifier.h